# tensor-unbind composite — cx16: chained unbind + tuple-unpack: extract three triangle vertices

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `tensor-unbind`, `unbind-tuple-unpack`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "tensor-unbind"
DD_ATOM_IDS = ["tensor-unbind", "unbind-tuple-unpack"]
DD_SUBTOPICS = ["Numpy: Indexing and selection", "PyTorch: Unbind tuple-unpack"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA part 1 represents triangle batches as `triangles: (NT, 3, 3)` — axis 1 enumerates the three vertices `(A, B, C)`, axis 2 is the spatial 3-vector. The natural way to pull out the three vertices is
  `A, B, C = t.unbind(triangles, dim=1)`
exercising both atoms in one line: `tensor-unbind` to split, `unbind-tuple-unpack` to bind the three returned tensors to named variables simultaneously.

Why this matters: the Python tuple-unpack idiom asserts at runtime that the LHS arity matches the RHS tuple length. If you wrote `A, B = t.unbind(triangles, dim=1)` you'd get `ValueError: too many values to unpack (expected 2)` — a structural assertion for free. That's why ARENA code uses unbind+tuple-unpack instead of three separate `triangles[:, i]` indexing calls.

### Composite Exercise — chained unbind + tuple-unpack: extract three triangle vertices

**Atoms exercised together**: `tensor-unbind`, `unbind-tuple-unpack`

Implement `cx16_triangle_normals(triangles)` that computes per-triangle unit-normal vectors via the chained unbind/tuple-unpack pattern.

Input: `triangles: (NT, 3, 3)` — axis 1 enumerates vertices `(A, B, C)`, axis 2 is the 3-vector coordinates.

1. **Unbind + tuple-unpack** the three vertices in ONE statement: `A, B, C = t.unbind(triangles, dim=1)`. Each is `(NT, 3)`.
2. Compute the edge vectors `e1 = B - A` and `e2 = C - A`, each `(NT, 3)`.
3. Compute the cross product `n = t.cross(e1, e2, dim=-1)` (the unnormalized normal).
4. Return the L2-normalized normal: divide by `t.linalg.vector_norm(n, dim=-1, keepdim=True)`. Result shape `(NT, 3)`.

The test asserts that the three unbound tensors map to vertices A, B, C in order — i.e. the tuple-unpack arity is correct and ordering matches axis 1.

In [ ]:
def cx16_triangle_normals(triangles):
    # Atoms A+B combined (tensor-unbind + unbind-tuple-unpack): one statement, three
    # named bindings. The tuple-unpack arity (3) asserts at runtime that axis 1 has size 3.
    A, B, C = t.unbind(triangles, dim=1)
    e1 = B - A
    e2 = C - A
    n = t.cross(e1, e2, dim=-1)
    return n / t.linalg.vector_norm(n, dim=-1, keepdim=True)


<details><summary>Show solution — cx16</summary>

```python
def cx16_triangle_normals(triangles):
    # Atoms A+B combined (tensor-unbind + unbind-tuple-unpack): one statement, three
    # named bindings. The tuple-unpack arity (3) asserts at runtime that axis 1 has size 3.
    A, B, C = t.unbind(triangles, dim=1)
    e1 = B - A
    e2 = C - A
    n = t.cross(e1, e2, dim=-1)
    return n / t.linalg.vector_norm(n, dim=-1, keepdim=True)
```

The single-line `A, B, C = t.unbind(triangles, dim=1)` is the canonical ARENA idiom. It does TWO things at once: (1) splits the size-3 vertex axis into a 3-tuple of `(NT, 3)` tensors, and (2) binds each element to a meaningfully-named variable. The structural assertion is free — Python raises `ValueError` if the tuple length doesn't match the LHS arity. Compare with `triangles[:, 0]; triangles[:, 1]; triangles[:, 2]`: three separate indexing calls, no name binding, no arity assertion. The unbind+unpack form is tighter and self-documenting.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx16'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx16',
        'subtopics': ["Numpy: Indexing and selection", "PyTorch: Unbind tuple-unpack"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()